In [14]:
import os
import json
import asyncio
import time
from pydantic import BaseModel, Field
from typing import List
from mistralai import Mistral
from tavily import TavilyClient
from dotenv import load_dotenv

# Chargement des clés
load_dotenv("cle.env", override=True)

client = Mistral(api_key=os.getenv("MISTRAL_API_KEY"))
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
print("✅ Moteurs Mistral et Tavily prêts.")

# Blacklist convertie en opérateurs de recherche
SOCIAL_BLACKLIST_QUERY = "-site:tiktok.com -site:facebook.com -site:instagram.com -site:x.com -site:twitter.com -site:reddit.com"

# --- SCHÉMAS DU ROUTEUR ---
class RouteurSchema(BaseModel):
    affirmation_propre: str
    run_stats: bool
    run_rhetorique: bool
    run_coherence_personnelle: bool
    run_contexte: bool
    a_verifier: bool

# --- SCHÉMAS DES AGENTS EXPERTS ---
class StatistiqueSchema(BaseModel):
    agent: str
    chiffre_cite: float
    chiffre_reel: float
    unite: str
    verdict_brut: str
    analyse_detaillee: str
    nom_source: str
    url_source: str


class RhetoriqueSchema(BaseModel):
    agent: str
    explication: str

class CoherenceSchema(BaseModel):
    agent: str
    explication: str

class ContexteSchema(BaseModel):
    agent: str
    analyse_detaillee: str
    nom_source: str
    url_source: str

class SourceSchema(BaseModel):
    organization: str
    url: str

class AnalysisSchema(BaseModel):
    summary: str
    sources: List[SourceSchema]

class ClaimSchema(BaseModel):
    text: str

class JugeSchema(BaseModel):
    claim: ClaimSchema
    analysis: AnalysisSchema
    overall_verdict: str = Field(description="Valeurs : accurate, inaccurate, exaggerated, misleading, partially_accurate")

✅ Moteurs Mistral et Tavily prêts.


In [15]:
async def agent_routeur(data):
    print(f"🚦 [Routeur] Analyse de l'entrée brute : '{data.get('affirmation', '')}'")
    
    prompt = f"""Tu es le cerveau analytique d'un système de fact-checking en direct.
    ENTRÉE BRUTE (Issue d'une transcription audio) : '{data.get('affirmation', '')}'

    MISSION 1 : CORRECTION PHONÉTIQUE ET TAMIS FACTUEL
    1. Corrige les erreurs de transcription évidentes (ex: "Le cuit" suivi d'un chiffre = "Le QI").
    2. ÉLIMINE le "bruit" : goûts personnels ("J'aime les chips"), opinions subjectives ou rhétorique vague.
    3. GARDE uniquement le "signal" : affirmations avec des chiffres, dates, noms propres.
    4. Si plusieurs faits sont présents, combine-les en une seule "affirmation_propre".
    5. Si aucun fait n'est détecté, mets "a_verifier" à false et "affirmation_propre" vide.

    MISSION 2 : ROUTAGE
    Si a_verifier est true, active les experts (true/false) :
    - 'run_stats' : Chiffres, pourcentages, économie, absolus ("aucun", "tous").
    - 'run_rhetorique' : Esquive d'une question posée.
    - 'run_coherence_personnelle' : Parle de son passé/déclarations.
    - 'run_contexte' : Évènements précis, lois. (PAS pour les stats pures).
    """
    
    # 🔥 LA NOUVEAUTÉ EST ICI : 
    # On utilise 'parse_async' au lieu de 'complete_async' 
    # Et on passe directement ta classe Pydantic en dur dans 'response_format'
    res = await client.chat.parse_async(
        model="mistral-small-latest", 
        messages=[{"role": "user", "content": prompt}], 
        response_format=RouteurSchema
    )
    
    # Plus besoin de json.loads() ! L'API renvoie directement un objet validé dans '.parsed'
    # On utilise model_dump() pour le retransformer en dictionnaire classique pour ton code Python
    return res.choices[0].message.parsed.model_dump()

In [16]:

# --- LISTES ET FILTRES (Simplifié) ---
SOCIAL_BLACKLIST = ["tiktok.com", "facebook.com", "instagram.com", "x.com", "twitter.com", "reddit.com", "quora.com", "pinterest.com"]

# --- AGENT 1 : STATISTIQUE (Version "Sources d'Élite") ---
async def agent_statistique(data):
    print(f"📊 [Agent Stat] Recherche RESTRINTE aux sources d'élite...")
    
    # On force la recherche sur les domaines que TU as choisis
    whitelist = " (site:insee.fr OR site:ccomptes.fr OR site:securite-sociale.fr OR site:lemonde.fr OR site:lesechos.fr OR site:ameli.fr)"
    query = f"montant réel dette sécurité sociale france 2024 2025" + whitelist
    
    try:
        search = tavily_client.search(
            query=query, 
            search_depth="advanced", 
            max_results=5
        )
        resultats = search.get('results', [])
        sources = "\n".join([f"Source: {r['url']}\nContenu: {r['content']}" for r in resultats])
        
        if not resultats:
             sources = "AUCUNE SOURCE OFFICIELLE TROUVÉE."
             
    except Exception as e:
        print(f"❌ [CRASH TAVILY] Erreur : {e}")
        sources = "Erreur technique."

    prompt = f"""Vérifie l'affirmation : '{data['affirmation']}'. 
    Sources institutionnelles : 
    {sources}

    MISSION DE VÉRIFICATION CRITIQUE : 
    1. Ne valide le chiffre QUE si la source parle explicitement de la "DETTE de la sécurité sociale". 
    2. Si la source parle d'autre chose (dépenses, autonomie, retraites), IGNORE-LA.
    3. Si le chiffre cité (30 milliards) ne correspond pas à la DETTE dans les sources, tu dois marquer "faux" et donner le vrai chiffre (ex: environ 17 milliards pour 2024).

    RENVOIE CE JSON :
    {{
      "agent": "statistique",
      "chiffre_cite": {data.get('chiffre_cite', 0.0)},
      "chiffre_reel": float,
      "unite": "milliards d'euros",
      "verdict_brut": "faux",
      "analyse_detaillee": "Soyez sec et factuel.",
      "nom_source": "...",
      "url_source": "..."
    }}
    """
    
    res = await client.chat.complete_async(
        model="mistral-small-latest", 
        messages=[{"role": "user", "content": prompt}], 
        response_format={"type": "json_object"}
    )
    return json.loads(res.choices[0].message.content)
# --- AGENT 4 : CONTEXTE ---
async def agent_contexte(data):
    print(f"📚 [Agent Contexte] Analyse factuelle détaillée...")
    
    query = f"{data['affirmation']} vrai ou faux explication France"

    try:
        search = tavily_client.search(
            query=query, 
            search_depth="basic", 
            max_results=3, 
            exclude_domains=SOCIAL_BLACKLIST
        )
        sources = "\n".join([f"Source: {r['url']}\nContenu: {r['content']}" for r in search.get('results', [])])
    except: 
        sources = "Aucun contexte."

    prompt = f"""Analyse le contexte de : '{data['affirmation']}'. 
    Sources : {sources}.
    ⚠️ ATTENTION CRITIQUE : Cette affirmation peut être un MENSONGE TOTAL.
    
    RÈGLES STRICTES :
    Fais une analyse factuelle pour expliquer le contexte RÉEL. Si les sources contredisent l'affirmation, explique la vraie situation.
    
    JSON: {{"agent": "contexte", "analyse_detaillee": "...", "nom_source": "...", "url_source": "..."}}"""
    
    res = await client.chat.parse_async(
        model="mistral-small-latest", 
        messages=[{"role": "user", "content": prompt}], 
        response_format=ContexteSchema
    )
    return res.choices[0].message.parsed.model_dump()

In [17]:
async def agent_editeur(affirmation_actuelle, rapports_agents):
    print("⚖️ [Juge] Application de la marge de 5% et formatage...")
    
    if not rapports_agents:
        return {
            "claim": {"text": affirmation_actuelle},
            "analysis": {
                "summary": "Aucun fait vérifiable détecté.",
                "sources": []
            },
            "overall_verdict": "accurate"
        }

    # 🔥 ON REMET TES INSTRUCTIONS QUI MARCHAINENT SI BIEN !
    prompt = f"""Tu es le Rédacteur en Chef. Ton but est de ne corriger que ce qui est vraiment faux, de manière naturelle.

    AFFIRMATION : "{affirmation_actuelle}"
    RAPPORTS DES EXPERTS : {json.dumps(rapports_agents, ensure_ascii=False)}

    RÈGLE D'OR (LA MARGE DES 5%) :
    1. Si un chiffre est cité, compare-le au chiffre réel des rapports.
    2. Si l'écart est <= 5%, le verdict (overall_verdict) est "accurate". On ne dit rien.
    3. Si l'écart est > 5%, le verdict est "inaccurate", "exaggerated", ou "misleading".
    
    CONSIGNE DE RÉDACTION (Dans analysis.summary) :
    - Écris UNE SEULE PHRASE en langage naturel, comme une précision journalistique.
    - Ajoute ton verdict en début de phrase suivi de :. Par exemple FAUX :, TROMPEUR :, EXAGÉRÉ :
    - NE PARLE JAMAIS de "marge", de "pourcentage d'erreur", de "fact-checking" ou "selon les sources".
    
    EXEMPLES ATTENDUS : 
    - "FAUX : On compte en réalité 330 000 personnes sans domicile fixe en France."
    - "EXAGÉRÉ : L'utilisation du glyphosate a augmenté de 15% en France cette année."
    """
    
    res = await client.chat.parse_async(
        model="mistral-small-latest", 
        messages=[{"role": "user", "content": prompt}], 
        response_format=JugeSchema
    )
    return res.choices[0].message.parsed.model_dump()

In [18]:
import time
import json
import asyncio

# --- EXÉCUTEUR PARALLÈLE (Mise à jour pour le nouveau Routeur) ---
async def executer_analyse_parallele(data):
    print(f"\n🚦 ENTRÉE BRUTE : '{data['affirmation']}'")
    
    # 1. Le routeur fait le tri
    routage = await agent_routeur(data)
    print("🧭 Décision du Routeur :")
    print(json.dumps(routage, indent=2, ensure_ascii=False))
    
    # 2. On arrête tout si ce n'est que du bruit
    if not routage.get("a_verifier"):
        print("🔇 Que du bruit ou avis subjectif détecté. Arrêt de l'analyse.")
        return [] 
        
    affirmation_propre = routage.get('affirmation_propre', data['affirmation'])
    print(f"✨ TEXTE NETTOYÉ : '{affirmation_propre}'")
    
    data_propre = data.copy()
    data_propre['affirmation'] = affirmation_propre 
    
    # 3. On lance les experts requis
    taches_a_lancer = []
    if routage.get("run_stats"): taches_a_lancer.append(agent_statistique(data_propre))
    if routage.get("run_rhetorique"): taches_a_lancer.append(agent_rhetorique(data_propre))
    if routage.get("run_coherence_personnelle"): taches_a_lancer.append(agent_coherence_personnelle(data_propre))
    if routage.get("run_contexte"): taches_a_lancer.append(agent_contexte(data_propre))
        
    if not taches_a_lancer:
        return [] 
    
    print(f"🚀 Lancement de {len(taches_a_lancer)} agent(s) expert(s)...")
    rapports = await asyncio.gather(*taches_a_lancer)
    return rapports

# --- TEST DU NOUVEAU WORKFLOW OBS ---
async def tester_pipeline_tv():
    print("🎬 DÉBUT DU TEST DE PRODUCTION OBS")
    print("="*70)
    
    # Les données d'entrée (avec du bruit exprès pour tester le filtre)
    data_instant_t = {
        "personne": "Gabriel Attal",
        "question_posee": "Quelle est la situation des comptes sociaux ?",
        "affirmation": "La dette de la sécurité sociale est de 30 milliards. J'aime le chocolat et les beignets."
    }

    start_time = time.perf_counter()
    
    # 1. Pipeline Multi-Agents (Routeur + Experts Web Search)
    rapports_bruts = await executer_analyse_parallele(data_instant_t)
    
    # 2. Le Juge (Pydantic / Structured Outputs)
    # Note : Le contexte_precedent a été retiré selon notre dernière version du Juge
    bandeau_tv = await agent_editeur(
        affirmation_actuelle=data_instant_t['affirmation'], 
        rapports_agents=rapports_bruts
    )
    
    end_time = time.perf_counter()
    
    print(f"\n⏱️ Temps total du Pipeline : {end_time - start_time:.2f}s")
    print("="*70)
    print("📦 JSON BRUT RENVOYÉ PAR LE JUGE (Ce que ton code reçoit) :")
    print(json.dumps(bandeau_tv, indent=2, ensure_ascii=False))
    print("="*70)
    
    # 3. Post-traitement Python pour OBS
    print("📺 SIMULATION DU TEXTE ENVOYÉ À L'ÉCRAN OBS :")
    
    verdict = bandeau_tv.get('overall_verdict', 'accurate')
    
    if verdict == "accurate":
        print("   [SILENCE RADIO] 🔇 -> L'affirmation est vraie, ou dans la marge de 5%.")
    else:
        # On extrait les données du JSON imbriqué
        texte_ecran = bandeau_tv['analysis']['summary']
        sources = bandeau_tv['analysis'].get('sources', [])
        
        # On formate la chaîne finale
        if sources:
            source_nom = sources[0].get('organization', 'Source Inconnue')
            flux_obs = f"{verdict.upper()} : {texte_ecran} (Source : {source_nom})"
        else:
            flux_obs = f"{verdict.upper()} : {texte_ecran}"
            
        print(f"   [AFFICHAGE] 🔴 -> {flux_obs}")
    print("="*70)

# Lancer la cellule
await tester_pipeline_tv()

🎬 DÉBUT DU TEST DE PRODUCTION OBS

🚦 ENTRÉE BRUTE : 'La dette de la sécurité sociale est de 30 milliards. J'aime le chocolat et les beignets.'
🚦 [Routeur] Analyse de l'entrée brute : 'La dette de la sécurité sociale est de 30 milliards. J'aime le chocolat et les beignets.'
🧭 Décision du Routeur :
{
  "affirmation_propre": "La dette de la sécurité sociale est de 30 milliards.",
  "run_stats": true,
  "run_rhetorique": false,
  "run_coherence_personnelle": false,
  "run_contexte": false,
  "a_verifier": true
}
✨ TEXTE NETTOYÉ : 'La dette de la sécurité sociale est de 30 milliards.'
🚀 Lancement de 1 agent(s) expert(s)...
📊 [Agent Stat] Recherche de chiffres sur le web...
⚖️ [Juge] Application de la marge de 5% et formatage...

⏱️ Temps total du Pipeline : 11.10s
📦 JSON BRUT RENVOYÉ PAR LE JUGE (Ce que ton code reçoit) :
{
  "claim": {
    "text": "La dette de la sécurité sociale est de 30 milliards. J'aime le chocolat et les beignets."
  },
  "analysis": {
    "summary": "ACCURATE : La pr

In [12]:
dataset_pipeline_expert = [
    # --- 5 EXEMPLES AVEC QUESTIONS (Q&A) ---
    {
        "p": "Gabriel Attal", "q": "Que faites-vous pour les sans-abris ?",
        "ctx": "La dignité humaine est au cœur de notre action. Nous avons ouvert 200 000 places d'hébergement d'urgence. L'État mobilise des moyens sans précédent cet hiver. Les préfets ont reçu des instructions claires pour ne laisser personne dehors. Nous avons investi 2 milliards d'euros dans le plan Logement d'Abord. Les associations sont nos partenaires quotidiens. La solidarité nationale doit jouer à plein. Nous ne laissons personne sur le côté. Le combat contre la pauvreté est une priorité.",
        "a": "Grâce à notre action, il n'y a plus aucun SDF en France aujourd'hui."
    },
    {
        "p": "Jordan Bardella", "q": "Quelle est votre position sur l'énergie ?",
        "ctx": "La France doit retrouver sa puissance électrique. Le marché européen nous impose des prix délirants qui ruinent les familles. Nous avons un parc nucléaire exceptionnel mais il a été saboté par l'idéologie. Il faut arrêter de fermer des centrales. La souveraineté énergétique est la base de l'indépendance nationale. Nos entreprises délocalisent à cause du coût de l'énergie. Le gaz russe a été remplacé par du gaz de schiste américain plus cher. C'est un non-sens écologique et économique.",
        "a": "Aujourd'hui, le nucléaire ne représente plus que 20% de notre électricité... enfin, 65%."
    },
    {
        "p": "Marine Tondelier", "q": "Faut-il interdire les pesticides ?",
        "ctx": "La santé des Français n'est pas négociable. L'eau que nous buvons est polluée par des molécules chimiques. Les agriculteurs sont les premières victimes de ces produits. On observe une chute dramatique de la biodiversité dans nos campagnes. Les abeilles disparaissent à un rythme alarmant. Le gouvernement recule sans cesse face aux lobbies industriels. Il faut accompagner la transition au lieu de la punir. L'agroécologie est la seule voie de survie pour nos sols.",
        "a": "L'utilisation du glyphosate a augmenté de 50% en France cette année."
    },
    {
        "p": "Bruno Le Maire", "q": "Où en sont les impôts ?",
        "ctx": "Nous avons fait le choix de la politique de l'offre. Baisser les impôts des entreprises permet de créer des emplois. Nous avons supprimé la taxe d'habitation pour tous les Français. La pression fiscale était trop forte dans ce pays. Nous voulons récompenser le travail et l'effort. Les prélèvements obligatoires ont commencé à refluer. C'est une trajectoire de long terme que nous tenons. La croissance revient grâce à cette stabilité fiscale.",
        "a": "Nous avons réussi à diviser par deux la dette de la France en cinq ans."
    },
    {
        "p": "Sandrine Rousseau", "q": "Pourquoi cibler les jets privés ?",
        "ctx": "On demande des efforts de sobriété aux Français modestes. On leur dit de baisser le chauffage à 19 degrés. Mais les ultra-riches continuent de brûler la planète en toute impunité. Un trajet en jet émet plus de CO2 qu'un Français moyen en un an. C'est une injustice climatique insupportable. Le gouvernement refuse de réguler ce secteur. L'écologie ne peut pas être punitive pour les pauvres et permissive pour les riches.",
        "a": "Mais la vraie question, c'est de savoir si le gouvernement va taxer les superprofits de Total !"
    },
    # --- 10 EXEMPLES DISCOURS BRUTS (SANS QUESTIONS) ---
    {
        "p": "Emmanuel Macron", "q": "",
        "ctx": "Nous vivons une période de grandes transformations mondiales. La France doit être aux avant-postes de la technologie. Nous investissons massivement dans l'IA et le quantique. L'école est le berceau de cette ambition future. Il faut réarmer notre éducation nationale. Le niveau en mathématiques doit devenir une priorité dès le CP. Nous avons déjà commencé à recruter davantage de professeurs. La formation continue sera le pilier de notre réforme.",
        "a": "Nous avons recruté 100 000 policiers supplémentaires cette année."
    },
    {
        "p": "Jean-Luc Mélenchon", "q": "",
        "ctx": "Le peuple a faim pendant que les riches se gavent. Les prix à la consommation étranglent les familles. On nous parle de croissance, mais la croissance de quoi ? De la misère ! Nous proposons le blocage des prix des produits de première nécessité. Il faut augmenter le SMIC immédiatement. La vie chère n'est pas une fatalité, c'est un choix politique. Les banques centrales ne font qu'aggraver la situation en montant les taux.",
        "a": "Le SMIC en France est actuellement à 2500 euros net."
    },
    {
        "p": "Marine Le Pen", "q": "",
        "ctx": "La sécurité est la première des libertés. Nos villes et nos villages sont livrés à une délinquance de plus en plus barbare. Le laxisme judiciaire encourage les récidivistes. Il faut rétablir l'autorité partout. Les forces de l'ordre doivent être soutenues par l'État. Nous avons besoin de 85 000 places de prison supplémentaires. Le lien entre immigration et insécurité n'est plus à démontrer pour personne. Les Français veulent vivre en paix.",
        "a": "Le taux de cambriolages a baissé de 40% en un an sous ce gouvernement."
    },
    {
        "p": "Rachida Dati", "q": "",
        "ctx": "La culture est le ciment de notre Nation. Elle ne doit pas être réservée aux élites parisiennes. Chaque village doit avoir accès à une programmation de qualité. Nous soutenons le patrimoine religieux et rural. C'est l'âme de nos territoires. Le pass culture est un outil formidable de démocratisation. Nous allons renforcer les moyens des bibliothèques de quartier. L'art doit entrer dans les écoles dès le plus jeune âge.",
        "a": "Le budget de la culture est le premier poste de dépense de l'État."
    },
    {
        "p": "Olivier Faure", "q": "",
        "ctx": "La réforme des retraites est une blessure sociale qui ne se refermera pas. Travailler jusqu'à 64 ans est une injustice pour ceux qui ont des carrières longues. Les femmes sont les grandes perdantes de cette loi. Nous proposons le retour à la retraite à 60 ans. La pénibilité doit être réellement prise en compte. On ne peut pas traiter un ouvrier comme un cadre. Le financement est possible en taxant les revenus du capital.",
        "a": "La réforme des retraites a été votée à l'unanimité par le Parlement."
    },
    {
        "p": "Eric Ciotti", "q": "",
        "ctx": "La France subit un dérapage budgétaire incontrôlé. Le déficit public atteint des sommets alarmants. Le gouvernement dépense l'argent qu'il n'a pas. Nos enfants hériteront d'une dette colossale. Il faut sabrer dans la dépense publique inutile. L'État doit se concentrer sur ses missions régaliennes. Nous sommes les champions du monde des prélèvements obligatoires. Il faut libérer les entreprises de ce poids mort administratif.",
        "a": "Le déficit public de la France est tombé à 1% du PIB en 2025."
    },
    {
        "p": "Manuel Bompard", "q": "",
        "ctx": "La Ve République est une monarchie présidentielle. Un seul homme décide de tout pour 67 millions de citoyens. L'article 49.3 est devenu l'outil de la brutalité politique. Nous devons passer à une VIe République parlementaire. Le référendum d'initiative citoyenne doit être instauré. Il faut redonner du pouvoir aux élus locaux. La démocratie ne peut pas se résumer à un vote tous les cinq ans. Le peuple doit pouvoir révoquer ses élus.",
        "a": "L'article 49.3 a été utilisé 100 fois cette année."
    },
    {
        "p": "Gérald Darmanin", "q": "",
        "ctx": "La lutte contre les stupéfiants est notre combat quotidien. Les opérations 'place nette' déstabilisent les réseaux. Nous saisissons des quantités records de cocaïne et de cannabis. Les points de deal reculent dans nos quartiers populaires. Nous protégeons les plus jeunes de cette économie souterraine. Les moyens de la gendarmerie sont renforcés. Nous créons 200 nouvelles brigades sur tout le territoire. L'ordre est la condition du progrès social.",
        "a": "Il y a plus de policiers à Paris qu'à New York."
    },
    {
        "p": "Yannick Jadot", "q": "",
        "ctx": "Le climat ne nous attend pas. Les rapports du GIEC sont de plus en plus alarmants. Nous devons sortir des énergies fossiles le plus vite possible. Les transports représentent un tiers de nos émissions. Le train doit devenir moins cher que l'avion. Nous devons isoler tous les bâtiments de ce pays. C'est bon pour la planète et pour le porte-monnaie. L'écologie est une opportunité industrielle majeure pour la France. La nature n'est pas une ressource infinie.",
        "a": "La France est le pays le plus polluant au monde par habitant."
    },
    {
        "p": "François Ruffin", "q": "",
        "ctx": "Le travail doit payer. Aujourd'hui, on peut travailler à temps plein et vivre sous le seuil de pauvreté. C'est indigne. La précarité explose dans les métiers du soin et du lien. Les aides à domicile sont les oubliées de la République. Elles font des kilomètres sans être remboursées. Il faut un salaire minimum européen. Les profits records ne ruissellent pas vers ceux qui produisent. La dignité ouvrière doit être remise au centre.",
        "a": "Le nombre de milliardaires en France a baissé de 50% sous ce mandat."
    }
]

In [13]:
import time
import json
import asyncio
import statistics # Pour calculer la moyenne facilement

async def run_benchmark_tv_production():
    print(f"⏱️  Lancement du test de charge OBS (15 tests)...")
    print("="*80)

    temps_execution = [] # Liste pour stocker le temps de chaque test

    for i, item in enumerate(dataset_pipeline_expert):
        try:
            start_time = time.perf_counter() # ⏱️ Début du chrono
            
            # 1. Analyse (Le routeur filtre, puis lance les experts Web Search)
            rapports = await executer_analyse_parallele({
                "personne": item['p'], 
                "question_posee": item['q'], 
                "affirmation": item['a']
            })
            
            # 2. Arbitrage du Juge (Attention à bien mettre les bons arguments selon ta fonction)
            # Si ton agent_editeur prend 2 arguments, enlève item['ctx']
            juge_output = await agent_editeur(item['a'], rapports)
            
            end_time = time.perf_counter() # ⏱️ Fin du chrono
            duree = end_time - start_time
            temps_execution.append(duree)
            
            print(f"\n🎬 TEST {i+1}/{len(dataset_pipeline_expert)} | {item['p']}")
            print(f"🗣️  AFFIRMATION : '{item['a']}'")
            
            # 3. Traitement du nouveau format Pydantic pour OBS
            verdict = juge_output.get('overall_verdict', 'accurate')
            
            if verdict != "accurate":
                # Extraction sécurisée des données imbriquées
                summary = juge_output['analysis']['summary']
                sources = juge_output['analysis'].get('sources', [])
                
                # Récupération de la première source si elle existe
                if sources:
                    source_nom = sources[0].get('organization', 'Source Inconnue')
                    chaine_obs = f"{verdict.upper()} : {summary} (Source : {source_nom})"
                else:
                    chaine_obs = f"{verdict.upper()} : {summary}"
                    
                print(f"✅ [ENVOI OBS] -> {chaine_obs}")
            else:
                print(f"🔇 [SILENCE OBS] (Phrase correcte ou dans la marge de 5%)")

            print(f"⏱️  Temps : {duree:.2f}s")
            print("-" * 80)

        except Exception as e:
            print(f"❌ Erreur au test {i+1} : {e}")
        
        await asyncio.sleep(2) # Sécurité pour ton i7 et éviter le Rate Limit Mistral

    # --- RÉSULTATS STATISTIQUES FINAUX ---
    if temps_execution:
        temps_moyen = statistics.mean(temps_execution)
        temps_max = max(temps_execution)
        
        print("\n" + "🚀"*15)
        print("📊 RÉSULTATS DES PERFORMANCES DU PIPELINE")
        print("🚀"*15)
        print(f"▶️ Temps d'exécution MOYEN   : {temps_moyen:.2f} secondes")
        print(f"▶️ Temps d'exécution MAXIMAL : {temps_max:.2f} secondes")
        print(f"▶️ Nombre de tests réussis   : {len(temps_execution)}/{len(dataset_pipeline_expert)}")
        print("="*80)
    else:
        print("⚠️ Aucun test n'a pu aller jusqu'au bout pour calculer les statistiques.")

# Pour lancer le test proprement :
await run_benchmark_tv_production()

⏱️  Lancement du test de charge OBS (15 tests)...

🚦 ENTRÉE BRUTE : 'Grâce à notre action, il n'y a plus aucun SDF en France aujourd'hui.'
🚦 [Routeur] Analyse de l'entrée brute : 'Grâce à notre action, il n'y a plus aucun SDF en France aujourd'hui.'
🧭 Décision du Routeur :
{
  "affirmation_propre": "il n'y a plus aucun SDF en France aujourd'hui",
  "run_stats": true,
  "run_rhetorique": false,
  "run_coherence_personnelle": false,
  "run_contexte": false,
  "a_verifier": true
}
✨ TEXTE NETTOYÉ : 'il n'y a plus aucun SDF en France aujourd'hui'
🚀 Lancement de 1 agent(s) expert(s)...
📊 [Agent Stat] Recherche profonde et tri strict...
🔍 [DEBUG] Meilleure source retenue : https://www.publicsenat.fr/actualites/sante/dette-de-la-securite-sociale-la-cour-des-comptes-alerte-sur-un-risque-de-crise-de-liquidite-des-2027
⚖️ [Juge] Application de la marge de 5% et formatage final imbriqué...

🎬 TEST 1/15 | Gabriel Attal
🗣️  AFFIRMATION : 'Grâce à notre action, il n'y a plus aucun SDF en France aujo

CancelledError: 